<a href="https://colab.research.google.com/github/sharmaAK30/Multilingual-NCERT-Doubt-Solver-using-OPEA-based-RAG-Pipeline/blob/main/NCERT_Doubt_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q fitz pytesseract pillow pdfplumber unsloth datasets
!sudo apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
import pandas as pd

try:
    df1 = pd.read_excel('/content/10 bio.xlsx')
    print("Set 1 columns:", df1.columns)
    all_data = pd.concat([df1], ignore_index=True)
except FileNotFoundError as e:
    print(f"Error: {e}. Please make sure the Excel files are uploaded to the /content/ directory.")
    all_data = pd.DataFrame() # Create an empty DataFrame to avoid NameError in the next cell

Set 1 columns: Index(['All green plants are ___.', 'Answer'], dtype='object')


In [ ]:
import json

if not all_data.empty:
    with open('biology_dataset.jsonl', 'w', encoding='utf-8') as f:
        for index, row in all_data.iterrows():
            prompt = "" # Initialize prompt
            response = "" # Initialize response
            # Case 1: First sheet columns
            if 'All green plants are ___. ' in row and 'Answer' in row:
                prompt = f"### Instruction:\nQ: {row['All green plants are ___. ']}\n\n### Response:"
                response = str(row['Answer'])

            # Write to file
            f.write(json.dumps({"text": f"{prompt}\n{response}"}, ensure_ascii=False) + "\n")
else:
    print("No data to process. Please check the file paths in the previous cell.")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="biology_dataset.jsonl", split="train")
print(f"Samples loaded: {len(dataset)}")
print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Samples loaded: 1044
{'text': '\n'}


In [ ]:
def format_and_tokenize(example):
    # Tokenize the text
    tokenized_input = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt" # Return PyTorch tensors
    )

    # For causal language modeling, the labels are the input_ids shifted by one
    # We need to clone input_ids to create labels and set padding tokens to -100
    labels = tokenized_input["input_ids"].clone()
    # Set padding tokens (-100) to be ignored in loss calculation
    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "input_ids": tokenized_input["input_ids"].squeeze(), # Remove batch dimension
        "attention_mask": tokenized_input["attention_mask"].squeeze(), # Remove batch dimension
        "labels": labels.squeeze() # Remove batch dimension
    }

tokenized_dataset = dataset.map(format_and_tokenize, batched=True, remove_columns=["text"])

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./biology-mistral-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)


In [ ]:
from unsloth import FastLanguageModel

# Load your quantized base model (8-bit, for example)
# model = AutoModelForCausalLM.from_pretrained(..., load_in_8bit=True)

# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    use_rslora=False
)


Unsloth 2025.8.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
def format_and_tokenize(example):
    tokenized_input = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=tokenizer.model_max_length,
    )

    input_ids = tokenized_input["input_ids"]
    attention_mask = tokenized_input["attention_mask"]

    # Make labels same shape as input_ids
    labels = input_ids.copy()
    labels = [(-100 if token == tokenizer.pad_token_id else token) for token in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [ ]:
tokenized_dataset = dataset.map(format_and_tokenize, remove_columns=["text"])


Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

In [ ]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",   # Unsloth can infer labels automatically
)

In [ ]:
model.save_pretrained("/content/biology-lora-model")
tokenizer.save_pretrained("/content/biology-lora-model")


('/content/biology-lora-model/tokenizer_config.json',
 '/content/biology-lora-model/special_tokens_map.json',
 '/content/biology-lora-model/chat_template.jinja',
 '/content/biology-lora-model/tokenizer.model',
 '/content/biology-lora-model/added_tokens.json',
 '/content/biology-lora-model/tokenizer.json')